In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

**Data Reading From Source**

In [0]:
df = spark.sql("select * from ete_cat.silver.customers_silver")

In [0]:
df.display()

customer_id,email,city,state,domains,full_name
C01991,craigbarajas@williamson.com,Sylviafort,MS,williamson.com,Sonia Matthews
C01992,pcross@hotmail.com,New Anthonybury,MT,hotmail.com,Andrew Turner
C01993,careyjohn@ortiz.com,Port Larrymouth,TN,ortiz.com,Daniel Flynn
C01994,barkertaylor@hampton.info,North Ericshire,NJ,hampton.info,Nicole Griffin
C01995,millerjodi@hotmail.com,Wolffort,FL,hotmail.com,Vincent Long
C01996,alan72@salas.com,Villarrealtown,WV,salas.com,David Warren
C01997,raydana@sanders.com,New Gabriel,NM,sanders.com,Amber Lawson
C01998,beckyjones@hotmail.com,Davidland,NH,hotmail.com,Ivan Jenkins
C01999,brandondiaz@mcdowell.biz,Stephaniechester,TX,mcdowell.biz,Kathleen Hodges
C02000,mcdowellkatie@yahoo.com,West Dianechester,DE,yahoo.com,Julie Smith


**Removing Duplicates**

In [0]:
df = df.dropDuplicates(subset=['customer_id'])

#**Dividing New vs Old Records**

In [0]:
if init_load_flag == 0:

    df_old = spark.sql('''select DimCustomerKey, customer_id, create_date, update_date
                        from ete_cat.gold.DimCustomers''')

else:

    df_old = spark.sql('''select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date
                        FROM ete_cat.silver.customers_silver where 1=0''')

In [0]:
df_old.display()

DimCustomerKey,customer_id,create_date,update_date
1,C00010,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
2,C00045,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
3,C00049,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
4,C00056,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
5,C00062,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
6,C00094,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
7,C00104,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
8,C00105,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
9,C00126,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
10,C00152,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z


**Renaming Columns of df_old**

In [0]:
df_old = df_old.withColumnRenamed("DimCustomerKey", "old_DimCustomerKey")\
                    .withColumnRenamed("customer_id", "old_customer_id")\
                    .withColumnRenamed("create_date", "old_create_date")\
                    .withColumnRenamed("update_date", "old_update_date")

**Applying Join with the Old Records**

In [0]:
df_join = df.join(df_old, df['customer_id'] == df_old['old_customer_id'], 'left')
df_join.display()


customer_id,email,city,state,domains,full_name,old_DimCustomerKey,old_customer_id,old_create_date,old_update_date
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,1,C00010,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
C00045,dkhan@hotmail.com,North Kara,OK,hotmail.com,Matthew Lee,2,C00045,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,3,C00049,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
C00056,cruzkristen@hotmail.com,Catherineberg,WA,hotmail.com,Tina Cantu,4,C00056,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
C00062,kimdennis@yahoo.com,Kaitlynburgh,MI,yahoo.com,Shelley Holland,5,C00062,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
C00094,rogersrandall@gonzales.org,Harrisonmouth,AK,gonzales.org,Hannah Olson,6,C00094,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
C00104,gloria15@yahoo.com,New Danielle,VA,yahoo.com,Patrick Meadows,7,C00104,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
C00105,christine52@kennedy.com,Crystalmouth,AL,kennedy.com,April Wright,8,C00105,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,9,C00126,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z
C00152,tara25@guzman-nelson.net,Ashleyside,ID,guzman-nelson.net,David Salazar,10,C00152,2026-09-16T02:35:48.273Z,2026-09-17T00:25:33.410Z


**Seperating New vs Old Records**

In [0]:
df_new = df_join.filter(df_join['old_DimCustomerKey'].isNull())

In [0]:
df_old = df_join.filter(df_join['old_DimCustomerKey'].isNotNull())

**Preparing df_old**

In [0]:
# Dropping all the columns which are not required
df_old = df_old.drop( 'old_customer_id', 'old_update_date')
# Renaming "old_DimCustomerkey" to "DimCustomerKey"

df_old = df_old.withColumnRenamed("old_DimCustomerKey", "DimCustomerKey")
# Renaming "old_create_date" column to "create_date"

df_old = df_old.withColumnRenamed("old_create_date", "create_date")
df_old = df_old.withColumn("create_date", to_timestamp(col("create_date")))
# Recreating "update_date" column with current timestamp

df_old = df_old.withColumn("update_date", current_timestamp())


In [0]:
df_old.display()

customer_id,email,city,state,domains,full_name,DimCustomerKey,create_date,update_date
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,1,2026-09-16T02:35:48.273Z,2026-09-17T01:27:50.762Z
C00045,dkhan@hotmail.com,North Kara,OK,hotmail.com,Matthew Lee,2,2026-09-16T02:35:48.273Z,2026-09-17T01:27:50.762Z
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,3,2026-09-16T02:35:48.273Z,2026-09-17T01:27:50.762Z
C00056,cruzkristen@hotmail.com,Catherineberg,WA,hotmail.com,Tina Cantu,4,2026-09-16T02:35:48.273Z,2026-09-17T01:27:50.762Z
C00062,kimdennis@yahoo.com,Kaitlynburgh,MI,yahoo.com,Shelley Holland,5,2026-09-16T02:35:48.273Z,2026-09-17T01:27:50.762Z
C00094,rogersrandall@gonzales.org,Harrisonmouth,AK,gonzales.org,Hannah Olson,6,2026-09-16T02:35:48.273Z,2026-09-17T01:27:50.762Z
C00104,gloria15@yahoo.com,New Danielle,VA,yahoo.com,Patrick Meadows,7,2026-09-16T02:35:48.273Z,2026-09-17T01:27:50.762Z
C00105,christine52@kennedy.com,Crystalmouth,AL,kennedy.com,April Wright,8,2026-09-16T02:35:48.273Z,2026-09-17T01:27:50.762Z
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,9,2026-09-16T02:35:48.273Z,2026-09-17T01:27:50.762Z
C00152,tara25@guzman-nelson.net,Ashleyside,ID,guzman-nelson.net,David Salazar,10,2026-09-16T02:35:48.273Z,2026-09-17T01:27:50.762Z


**Preparing df_new**

In [0]:
# Dropping all the columns which are not required

df_new = df_new.drop('old_DimCustomerKey', 'old_customer_id', 'old_update_date', 'old_create_date')

# Recreating "update_date", "current_date" columns with current timestamp

df_new = df_new.withColumn("update_date", current_timestamp())
df_new = df_new.withColumn("create_date", current_timestamp())

In [0]:
df_new.display()

customer_id,email,city,state,domains,full_name,update_date,create_date


**Surrogate Key -From 1**

In [0]:
df_new = df_new.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1))

**Adding Max Surrogate Key**

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0

else:
    df_maxsur = spark.sql("select max(DimCustomerKey) as max_surrogate_key from ete_cat.gold.DimCustomers")
    #Converting df_maxsur to max_surrogate key variable
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']

In [0]:
df_new = df_new.withColumn("DimCustomerKey",lit(max_surrogate_key)+col("DimCustomerKey"))

**Union of df_old and df_new**

In [0]:
df_final = df_new.unionByName(df_old)

**SCD Type-1**

In [0]:
from delta.tables import DeltaTable

In [0]:
if (spark.catalog.tableExists("ete_cat.gold.DimCustomers")):

    dlt_obj = DeltaTable.forPath(spark,"abfss://gold@eteproj.dfs.core.windows.net/DimCustomers")

    dlt_obj.alias("trg").merge(df_final.alias("src"),"trg.DimCustomerKey = src.DimCustomerKey")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_final.write.mode("overwrite")\
        .format("delta")\
        .option("path","abfss://gold@eteproj.dfs.core.windows.net/DimCustomers")\
        .saveAsTable("ete_cat.gold.DimCustomers")